# CNN baseline backbone + DQN Active Learning：OASIS1-only + unlabeled pool


```python
CNN_ARCH = 'current'      # Proposed lightweight 2.5D CNN
CNN_ARCH = 'basic_cnn'    # BasicCNN baseline
CNN_ARCH = 'resnet18'     # ResNet18 baseline
CNN_ARCH = 'densenet121'  # DenseNet121 baseline
```


In [1]:
from pathlib import Path
import subprocess, sys, json, shlex
import pandas as pd

PROJECT_DIR = Path.cwd()
SCRIPT = PROJECT_DIR / 'train_cnn_dqn_de.py'
BUILD_MANIFEST = PROJECT_DIR / 'build_manifest.py'

OASIS1_ROOT = '/home/frostxia/Desktop/training_data/OASIS1/RAW'
OASIS1_CLINICAL = PROJECT_DIR / 'data_tables' / 'oasis_cross-sectional.xlsx'

MANIFEST_ROOT = PROJECT_DIR / 'outputs_cv_manifests_al'
OUTPUT_ROOT = PROJECT_DIR / 'outputs_dqn_best_checkpoint'
CACHE_ROOT = PROJECT_DIR / 'cache_dqn_best_checkpoint'
for p in [MANIFEST_ROOT, OUTPUT_ROOT, CACHE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, cwd=PROJECT_DIR):
    cmd = [str(x) for x in cmd]
    print('\n$ ' + ' '.join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, check=True, cwd=str(cwd))

print('PROJECT_DIR      =', PROJECT_DIR)
print('BUILD_MANIFEST   =', BUILD_MANIFEST, '  # Oasis1 manifest builder script')
print('SCRIPT           =', SCRIPT, '  # Training script')
print('MANIFEST_ROOT    =', MANIFEST_ROOT)
print('OUTPUT_ROOT      =', OUTPUT_ROOT)
print('CACHE_ROOT       =', CACHE_ROOT)
assert BUILD_MANIFEST.exists(), f'Missing build_manifest.py: {BUILD_MANIFEST}'
assert SCRIPT.exists(), f'Missing train_cnn_dqn_de.py: {SCRIPT}'


PROJECT_DIR      = /home/frostxia/Desktop/mri_test/AI_DL8
BUILD_MANIFEST   = /home/frostxia/Desktop/mri_test/AI_DL8/build_manifest.py   # Oasis1 manifest builder script
SCRIPT           = /home/frostxia/Desktop/mri_test/AI_DL8/train_cnn_dqn_de.py   # Training script
MANIFEST_ROOT    = /home/frostxia/Desktop/mri_test/AI_DL8/outputs_cv_manifests_al
OUTPUT_ROOT      = /home/frostxia/Desktop/mri_test/AI_DL8/outputs_dqn_best_checkpoint
CACHE_ROOT       = /home/frostxia/Desktop/mri_test/AI_DL8/cache_dqn_best_checkpoint


## 1. Build manifest


In [2]:
RUN_BUILD_MANIFESTS = True
DROP_OASIS1_UNLABELED = True  # True = remove unlabeled samples from manifest; False = keep unlabeled samples in manifest

if RUN_BUILD_MANIFESTS:
    cmd = [
        sys.executable, BUILD_MANIFEST,
        '--oasis1-root', OASIS1_ROOT,
        '--oasis1-clinical', OASIS1_CLINICAL,
        '--output-dir', MANIFEST_ROOT / 'oasis1',
        '--seed', '42',
    ]
    if DROP_OASIS1_UNLABELED:
        cmd.append('--drop-oasis1-unlabeled')
    run_cmd(cmd)



$ /home/frostxia/miniconda3/envs/oasis_sfcn/bin/python /home/frostxia/Desktop/mri_test/AI_DL8/build_manifest.py --oasis1-root /home/frostxia/Desktop/training_data/OASIS1/RAW --oasis1-clinical /home/frostxia/Desktop/mri_test/AI_DL8/data_tables/oasis_cross-sectional.xlsx --output-dir /home/frostxia/Desktop/mri_test/AI_DL8/outputs_cv_manifests_al/oasis1 --seed 42 --drop-oasis1-unlabeled
OASIS1 matched images: 235 (unlabeled pool candidates: 0)

Manifest summary
         split  n_images  n_subjects         label_counts  n_labeled_images  n_unlabeled_images labeled_label_counts  dataset_counts
      combined       235         235 {'0': 135, '1': 100}             235.0                 0.0 {'0': 135, '1': 100} {'OASIS1': 235}
         train       164         164   {'0': 94, '1': 70}             164.0                 0.0   {'0': 94, '1': 70} {'OASIS1': 164}
           val        35          35   {'0': 20, '1': 15}              35.0                 0.0   {'0': 20, '1': 15}  {'OASIS1': 35}
    

## 2. Review manifest


In [3]:
MANIFESTS = {
    'oasis1': MANIFEST_ROOT / 'oasis1' / 'manifests' / 'al_manifest.csv',
}

def show_manifest(name, path):
    path = Path(path)
    assert path.exists(), f'Manifest not found: {path}'
    df = pd.read_csv(path)
    print(f'\n{name}: {path}')
    print('images:', len(df), 'subjects:', df['subject_key'].nunique())
    if 'split' in df.columns:
        print('\nsplit counts:')
        print(df['split'].value_counts(dropna=False))
    if 'is_labeled' in df.columns:
        print('\nis_labeled counts:')
        print(df['is_labeled'].value_counts(dropna=False))
    if 'label' in df.columns:
        print('\nlabel counts:')
        print(df['label'].value_counts(dropna=False).sort_index())
    return df

for name, path in MANIFESTS.items():
    show_manifest(name, path)



oasis1: /home/frostxia/Desktop/mri_test/AI_DL8/outputs_cv_manifests_al/oasis1/manifests/al_manifest.csv
images: 235 subjects: 235

split counts:
split
train    164
test      36
val       35
Name: count, dtype: int64

is_labeled counts:
is_labeled
True    235
Name: count, dtype: int64

label counts:
label
0    135
1    100
Name: count, dtype: int64


## 3. Set train parameters


In [ ]:
DATASET_TO_RUN = 'oasis1'       # OASIS1-only
DEVICE = 'cuda' # 'cuda' or 'cpu' or 'mps' (Apple Silicon)
USE_AMP = True # only effective if DEVICE is 'cuda' or 'mps'; has no effect if DEVICE is 'cpu'
SEED = 42 # random seed for reproducibility

IMAGE_SIZE = (128, 128)

# Data input mode:
#   '2.5d' = sagittal + coronal + axial 
#   '2d'   = single plane (see PLANE and N_SLICES)
INPUT_MODE = '2.5d'  # '2.5d' / '2d'

# 2.5D input_channels = 3 * SLICES_PER_PLANE
SLICES_PER_PLANE = 15

# 2D input: which plane and how many slices
PLANE = 'axial'      # 'axial' / 'coronal' / 'sagittal'
N_SLICES = 15

# SLICE_FRACTION_MIN is the minimum fraction of slices to use from each scan, and SLICE_FRACTION_MAX is the maximum fraction.
SLICE_FRACTION_MIN = 0.15
SLICE_FRACTION_MAX = 0.85

# 5 for 5 fold cross-validation; 1 to strictly use the train/val/test splits in the manifest
FOLDS = 5
RESPECT_MANIFEST_SPLITS = False

BEST_CHECKPOINT_METRIC = 'composite'

# change the cnn backbone here: 'current' / 'basic_cnn' / 'resnet18' / 'densenet121'
CNN_ARCH = 'current'
# resnet18/densenet121 use pretrained weights by default; set to False if your environment cannot download/load weights.
PRETRAINED_BACKBONE = True


INITIAL_LABEL_FRACTION = 0.25
CYCLES = 5
# Budget per cycle: number of samples to annotate in each active learning cycle. Ignored if UNTIL_UNLABELED_EXHAUSTED=True.
BUDGET_PER_CYCLE = 16
# Maximum number of annotations to acquire across all cycles. Set to -1 for no limit. Ignored if UNTIL_UNLABELED_EXHAUSTED=True.
MAX_ANNOTATIONS = -1
# if True, each cycle will keep annotating until the unlabeled pool is exhausted; if False, each cycle will annotate at most BUDGET_PER_CYCLE samples.
UNTIL_UNLABELED_EXHAUSTED = True

# Save test heatmaps during training for qualitative analysis. 
SAVE_TEST_HEATMAPS_DURING_TRAINING = True
# Which checkpoint to use for test heatmap generation: 'best' (best val metric), 'final' (final epoch), or 'last' (last epoch).
HEATMAP_CHECKPOINT = 'best'      
# Which heatmap method(s) to use: 'input-gradient', 'branch-gradcam', or 'both' (if both, input-gradient will be saved as .npy and branch-gradcam will be saved as .png).
HEATMAP_METHOD = 'both'          
# For heatmap visualization, which target to use: 'pred' (model's predicted probability), 'true' (ground truth label), or 'demented' (the "demented")
HEATMAP_TARGET = 'pred'         
# Heatmap overlay alpha (transparency) for visualization; only affects .png outputs, not .npy.
HEATMAP_ALPHA = 0.45
# Heatmap image DPI for saving .png files; higher means higher resolution and larger file size.
HEATMAP_DPI = 180
# Save heatmaps as .npy files (for input-gradient) and/or .png files (for branch-gradcam); if False, only display during training without saving.
HEATMAP_SAVE_NPY = False
HEATMAP_SAVE_ALL_SLICE_PNGS = False
# The fractions of samples to display heatmaps for during training. 
HEATMAP_DISPLAY_FRACTIONS = (0.5, 0.5, 0.5)
HEATMAP_DISPLAY_INDICES = None   # Set specific sample indices to display heatmaps for during training; if None, use HEATMAP_DISPLAY_FRACTIONS 

# set to True to use the optimal threshold on the validation set for computing metrics and selecting best checkpoint; if False, use the default 0.5 threshold.
USE_VAL_OPTIMAL_THRESHOLD = False # use 0.5 threshold by default

COMMON_TRAIN_ARGS = [
    '--image-size', str(IMAGE_SIZE[0]), str(IMAGE_SIZE[1]),
    '--slices-per-plane', str(SLICES_PER_PLANE),
    '--slice-fraction-min', str(SLICE_FRACTION_MIN),
    '--slice-fraction-max', str(SLICE_FRACTION_MAX),
    '--seed', str(SEED),
    '--cnn-arch', CNN_ARCH,
    '--device', DEVICE,
    '--folds', str(FOLDS),
    '--initial-label-fraction', str(INITIAL_LABEL_FRACTION),
    '--cycles', str(CYCLES),
    '--budget-per-cycle', str(BUDGET_PER_CYCLE),
    '--max-annotations', str(MAX_ANNOTATIONS),
    '--best-checkpoint-metric', BEST_CHECKPOINT_METRIC,
    '--loss-name', 'focal',
    '--lr', '0.002',
    '--batch-size', '32',
    '--initial-epochs', '78',
    '--retrain-epochs', '40',
    '--cnn-stacks', '3',
    '--q-mlp-layers', '4',
    '--scope-alpha', '0.2',
    '--scope-weight', '0.05',
    '--advantage-clip', '5.0',
    '--q-policy-temperature', '1.0',
    '--dqn-grad-clip', '5.0',
    '--reward-mode', 'centroid',
]

if CNN_ARCH in {'resnet18', 'densenet121'}:
    COMMON_TRAIN_ARGS.append('--pretrained-backbone' if PRETRAINED_BACKBONE else '--no-pretrained-backbone')

if FOLDS == 1:
    COMMON_TRAIN_ARGS.append('--respect-manifest-splits' if RESPECT_MANIFEST_SPLITS else '--no-respect-manifest-splits')
if USE_AMP:
    COMMON_TRAIN_ARGS.append('--amp')

if USE_VAL_OPTIMAL_THRESHOLD:
    COMMON_TRAIN_ARGS.append('--use-val-optimal-threshold')

if UNTIL_UNLABELED_EXHAUSTED:
    COMMON_TRAIN_ARGS.append('--until-unlabeled-exhausted')

if SAVE_TEST_HEATMAPS_DURING_TRAINING:
    COMMON_TRAIN_ARGS += [
        '--save-test-heatmaps',
        '--heatmap-method', HEATMAP_METHOD,
        '--heatmap-target', HEATMAP_TARGET,
        '--heatmap-alpha', str(HEATMAP_ALPHA),
        '--heatmap-dpi', str(HEATMAP_DPI),
        '--heatmap-display-fractions', *[str(x) for x in HEATMAP_DISPLAY_FRACTIONS],
    ]
    if HEATMAP_DISPLAY_INDICES is not None:
        COMMON_TRAIN_ARGS += ['--heatmap-display-indices', *[str(x) for x in HEATMAP_DISPLAY_INDICES]]
    if HEATMAP_SAVE_NPY:
        COMMON_TRAIN_ARGS.append('--heatmap-save-npy')
    if HEATMAP_SAVE_ALL_SLICE_PNGS:
        COMMON_TRAIN_ARGS.append('--heatmap-save-all-slice-pngs')



## 4. Start Training


In [5]:
def run_al_experiment(name, manifest_csv):
    manifest_csv = Path(manifest_csv)
    assert manifest_csv.exists(), f'Manifest not found: {manifest_csv}'

    out_dir = OUTPUT_ROOT / f'{name}_{INPUT_MODE}_{CNN_ARCH}'
    cache_npz = CACHE_ROOT / f'{name}_{INPUT_MODE}_{PLANE}_img{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}_2d{N_SLICES}_25d{SLICES_PER_PLANE}_{SLICE_FRACTION_MIN}_{SLICE_FRACTION_MAX}.npz'

    cmd = [
        sys.executable, SCRIPT,
        '--manifest-csv', manifest_csv,
        '--output-dir', out_dir,
        '--cache-npz', cache_npz,
    ] + COMMON_TRAIN_ARGS

    run_cmd(cmd)
    return out_dir

RUN_TRAINING = True
if RUN_TRAINING:
    TRAIN_OUT = run_al_experiment(DATASET_TO_RUN, MANIFESTS[DATASET_TO_RUN])
    print('TRAIN_OUT =', TRAIN_OUT)



$ /home/frostxia/miniconda3/envs/oasis_sfcn/bin/python /home/frostxia/Desktop/mri_test/AI_DL8/train_cnn_dqn_de.py --manifest-csv /home/frostxia/Desktop/mri_test/AI_DL8/outputs_cv_manifests_al/oasis1/manifests/al_manifest.csv --output-dir /home/frostxia/Desktop/mri_test/AI_DL8/outputs_dqn_best_checkpoint/oasis1_2.5d_current --cache-npz /home/frostxia/Desktop/mri_test/AI_DL8/cache_dqn_best_checkpoint/oasis1_2.5d_axial_img128x128_2d15_25d15_0.15_0.85.npz --image-size 128 128 --slices-per-plane 15 --slice-fraction-min 0.15 --slice-fraction-max 0.85 --seed 42 --cnn-arch current --device cuda --folds 5 --initial-label-fraction 0.25 --cycles 5 --budget-per-cycle 16 --max-annotations -1 --best-checkpoint-metric composite --loss-name focal --lr 0.002 --batch-size 32 --initial-epochs 78 --retrain-epochs 40 --cnn-stacks 3 --q-mlp-layers 4 --scope-alpha 0.2 --scope-weight 0.05 --advantage-clip 5.0 --q-policy-temperature 1.0 --dqn-grad-clip 5.0 --reward-mode centroid --amp --until-unlabeled-exhaus

load MRI 2.5D multi-slice inputs: 100%|██████████| 235/235 [00:20<00:00, 11.23it/s]


Saved cache: /home/frostxia/Desktop/mri_test/AI_DL8/cache_dqn_best_checkpoint/oasis1_2.5d_axial_img128x128_2d15_25d15_0.15_0.85.npz
Loaded: input_mode=2.5d, X=(235, 45, 128, 128), labeled_y=[135, 100], unlabeled_pool=0, subjects=235

================ Fold 1/5 ================
{
  "train": {
    "n_images": 160,
    "n_subjects": 160,
    "n_labeled_images": 160,
    "n_unlabeled_images": 0,
    "label_counts_images": {
      "0": 92,
      "1": 68
    },
    "label_counts_subjects": {
      "0": 92,
      "1": 68
    },
    "dataset_counts": {
      "OASIS1": 160
    }
  },
  "val": {
    "n_images": 28,
    "n_subjects": 28,
    "n_labeled_images": 28,
    "n_unlabeled_images": 0,
    "label_counts_images": {
      "0": 16,
      "1": 12
    },
    "label_counts_subjects": {
      "0": 16,
      "1": 12
    },
    "dataset_counts": {
      "OASIS1": 28
    }
  },
  "test": {
    "n_images": 47,
    "n_subjects": 47,
    "n_labeled_images": 47,
    "n_unlabeled_images": 0,
    "label_c

  Test metrics: {
  "accuracy": 0.7872340425531915,
  "balanced_accuracy": 0.788888888888889,
  "auc": 0.8388888888888888,
  "precision": 0.7272727272727273,
  "recall": 0.8,
  "f1": 0.7619047619047619,
  "sensitivity": 0.7999999999999601,
  "specificity": 0.777777777777749,
  "gmean": 0.7888106377465812,
  "threshold": 0.5,
  "tn": 21,
  "fp": 6,
  "fn": 4,
  "tp": 16
}

================ Fold 2/5 ================
{
  "train": {
    "n_images": 160,
    "n_subjects": 160,
    "n_labeled_images": 160,
    "n_unlabeled_images": 0,
    "label_counts_images": {
      "0": 92,
      "1": 68
    },
    "label_counts_subjects": {
      "0": 92,
      "1": 68
    },
    "dataset_counts": {
      "OASIS1": 160
    }
  },
  "val": {
    "n_images": 28,
    "n_subjects": 28,
    "n_labeled_images": 28,
    "n_unlabeled_images": 0,
    "label_counts_images": {
      "0": 16,
      "1": 12
    },
    "label_counts_subjects": {
      "0": 16,
      "1": 12
    },
    "dataset_counts": {
      "OASIS

  Test metrics: {
  "accuracy": 0.574468085106383,
  "balanced_accuracy": 0.6231481481481481,
  "auc": 0.7055555555555555,
  "precision": 0.5,
  "recall": 0.95,
  "f1": 0.6551724137931034,
  "sensitivity": 0.9499999999999525,
  "specificity": 0.29629629629628534,
  "gmean": 0.5305482838361246,
  "threshold": 0.5,
  "tn": 8,
  "fp": 19,
  "fn": 1,
  "tp": 19
}

================ Fold 3/5 ================
{
  "train": {
    "n_images": 160,
    "n_subjects": 160,
    "n_labeled_images": 160,
    "n_unlabeled_images": 0,
    "label_counts_images": {
      "0": 92,
      "1": 68
    },
    "label_counts_subjects": {
      "0": 92,
      "1": 68
    },
    "dataset_counts": {
      "OASIS1": 160
    }
  },
  "val": {
    "n_images": 28,
    "n_subjects": 28,
    "n_labeled_images": 28,
    "n_unlabeled_images": 0,
    "label_counts_images": {
      "0": 16,
      "1": 12
    },
    "label_counts_subjects": {
      "0": 16,
      "1": 12
    },
    "dataset_counts": {
      "OASIS1": 28
    }

  Test metrics: {
  "accuracy": 0.7446808510638298,
  "balanced_accuracy": 0.7453703703703703,
  "auc": 0.8018518518518518,
  "precision": 0.6818181818181818,
  "recall": 0.75,
  "f1": 0.7142857142857143,
  "sensitivity": 0.7499999999999626,
  "specificity": 0.7407407407407134,
  "gmean": 0.7453559924998975,
  "threshold": 0.5,
  "tn": 20,
  "fp": 7,
  "fn": 5,
  "tp": 15
}

================ Fold 4/5 ================
{
  "train": {
    "n_images": 160,
    "n_subjects": 160,
    "n_labeled_images": 160,
    "n_unlabeled_images": 0,
    "label_counts_images": {
      "0": 92,
      "1": 68
    },
    "label_counts_subjects": {
      "0": 92,
      "1": 68
    },
    "dataset_counts": {
      "OASIS1": 160
    }
  },
  "val": {
    "n_images": 28,
    "n_subjects": 28,
    "n_labeled_images": 28,
    "n_unlabeled_images": 0,
    "label_counts_images": {
      "0": 16,
      "1": 12
    },
    "label_counts_subjects": {
      "0": 16,
      "1": 12
    },
    "dataset_counts": {
      "OA

  Test metrics: {
  "accuracy": 0.6170212765957447,
  "balanced_accuracy": 0.575925925925926,
  "auc": 0.7592592592592592,
  "precision": 0.6,
  "recall": 0.3,
  "f1": 0.4,
  "sensitivity": 0.299999999999985,
  "specificity": 0.8518518518518203,
  "gmean": 0.5055250296034147,
  "threshold": 0.5,
  "tn": 23,
  "fp": 4,
  "fn": 14,
  "tp": 6
}

================ Fold 5/5 ================
{
  "train": {
    "n_images": 160,
    "n_subjects": 160,
    "n_labeled_images": 160,
    "n_unlabeled_images": 0,
    "label_counts_images": {
      "0": 92,
      "1": 68
    },
    "label_counts_subjects": {
      "0": 92,
      "1": 68
    },
    "dataset_counts": {
      "OASIS1": 160
    }
  },
  "val": {
    "n_images": 28,
    "n_subjects": 28,
    "n_labeled_images": 28,
    "n_unlabeled_images": 0,
    "label_counts_images": {
      "0": 16,
      "1": 12
    },
    "label_counts_subjects": {
      "0": 16,
      "1": 12
    },
    "dataset_counts": {
      "OASIS1": 28
    }
  },
  "test": {


  Test metrics: {
  "accuracy": 0.7021276595744681,
  "balanced_accuracy": 0.6953703703703704,
  "auc": 0.8518518518518519,
  "precision": 0.65,
  "recall": 0.65,
  "f1": 0.65,
  "sensitivity": 0.6499999999999676,
  "specificity": 0.7407407407407134,
  "gmean": 0.6938886664886809,
  "threshold": 0.5,
  "tn": 20,
  "fp": 7,
  "fn": 7,
  "tp": 13
}

================ CV fold metrics ================
 fold  labeled_count  best_val_threshold  best_cycle  best_val_score  final_labeled_count  final_queried_count  accuracy  balanced_accuracy      auc  precision  recall       f1  sensitivity  specificity    gmean  threshold  tn  fp  fn  tp
    1            104                 0.5           4        0.739500                  160                  120  0.787234           0.788889 0.838889   0.727273    0.80 0.761905         0.80     0.777778 0.788811        0.5  21   6   4  16
    2             56                 0.5           1        0.714247                  160                  120  0.574468  

## 5. Produce heatmap only


In [6]:
RUN_EXTERNAL_HEATMAPS = False

def run_heatmaps_only(name, manifest_csv, out_dir):
    manifest_csv = Path(manifest_csv)
    out_dir = Path(out_dir)
    assert manifest_csv.exists(), f'Manifest not found: {manifest_csv}'
    assert out_dir.exists(), f'Training output dir not found: {out_dir}'

    cache_npz = CACHE_ROOT / f'{name}_{INPUT_MODE}_{PLANE}_img{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}_2d{N_SLICES}_25d{SLICES_PER_PLANE}_{SLICE_FRACTION_MIN}_{SLICE_FRACTION_MAX}.npz'
    cmd = [
        sys.executable, SCRIPT,
        '--manifest-csv', manifest_csv,
        '--output-dir', out_dir,
        '--cache-npz', cache_npz,
        '--input-mode', INPUT_MODE,
        '--plane', PLANE,
        '--n-slices', str(N_SLICES),
        '--heatmap-only',
        '--heatmap-checkpoint', HEATMAP_CHECKPOINT,
        '--heatmap-method', HEATMAP_METHOD,
        '--heatmap-target', HEATMAP_TARGET,
        '--heatmap-alpha', str(HEATMAP_ALPHA),
        '--heatmap-dpi', str(HEATMAP_DPI),
        '--heatmap-display-fractions', *[str(x) for x in HEATMAP_DISPLAY_FRACTIONS],
        '--device', DEVICE,
    ]
    if HEATMAP_DISPLAY_INDICES is not None:
        cmd += ['--heatmap-display-indices', *[str(x) for x in HEATMAP_DISPLAY_INDICES]]
    if HEATMAP_SAVE_NPY:
        cmd.append('--heatmap-save-npy')
    if HEATMAP_SAVE_ALL_SLICE_PNGS:
        cmd.append('--heatmap-save-all-slice-pngs')
    run_cmd(cmd)

if RUN_EXTERNAL_HEATMAPS:
    # If already trained in current session, directly use TRAIN_OUT; otherwise manually specify an existing output directory. --- IGNORE ---
    heatmap_out = Path(globals().get('TRAIN_OUT', OUTPUT_ROOT / f'{DATASET_TO_RUN}_{CNN_ARCH}'))
    run_heatmaps_only(DATASET_TO_RUN, MANIFESTS[DATASET_TO_RUN], heatmap_out)



## 6. Output check


In [7]:
# 快速查看 best checkpoint 和 test 指标
if 'TRAIN_OUT' in globals():
    for fold_dir in sorted(Path(TRAIN_OUT).glob('fold_*')):
        print('\n', fold_dir)
        for rel in ['best_checkpoint.json', 'test_metrics.json', 'manifests/unlabeled_pool_manifest.csv']:
            p = fold_dir / rel
            print(' ', rel, 'exists=', p.exists())
            if p.exists() and p.suffix == '.json':
                print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False)[:2000])



 /home/frostxia/Desktop/mri_test/AI_DL8/outputs_dqn_best_checkpoint/oasis1_2.5d_current/fold_01
  best_checkpoint.json exists= True
{
  "best_cycle": 4,
  "best_val_score": 0.7394999999999892,
  "best_checkpoint_metric": "composite",
  "best_val_threshold": 0.5,
  "best_labeled_count": 104,
  "final_labeled_count": 160,
  "final_queried_count": 120,
  "note": "classifier_final.pt and dqn_final.pt are aliases of the best validation checkpoint; *_last.pt keeps the last AL-cycle state."
}
  test_metrics.json exists= True
{
  "accuracy": 0.7872340425531915,
  "balanced_accuracy": 0.788888888888889,
  "auc": 0.8388888888888888,
  "precision": 0.7272727272727273,
  "recall": 0.8,
  "f1": 0.7619047619047619,
  "sensitivity": 0.7999999999999601,
  "specificity": 0.777777777777749,
  "gmean": 0.7888106377465812,
  "threshold": 0.5,
  "tn": 21,
  "fp": 6,
  "fn": 4,
  "tp": 16
}
  manifests/unlabeled_pool_manifest.csv exists= True

 /home/frostxia/Desktop/mri_test/AI_DL8/outputs_dqn_best_checkp